# 第08回 ソルバ2：輸送計画と簡易ドッキング

**オンデマンド・小レポートなし。** PDFとこのノートブックを並べ、上から実行します。練習は答えを見る前に予想し、実行結果と比べてください。途中で保存して再開できます。


## 1. 準備
「ファイル → ドライブにコピーを保存」で自分のコピーを作り、上から順に実行します。CPUを使います。再接続で新しい環境になったら準備セルから実行します。


In [ ]:
%pip -q install "pulp==3.3.0" "highspy==1.12.0"

In [ ]:
import pulp
print("PuLP", pulp.__version__)
print("利用可能なソルバ:", pulp.listSolvers(onlyAvailable=True))

## 2. 前回の工場を確認する
製品1・2の連続的な生産量を求めます。条件は06・07と同じです。

In [ ]:
model = pulp.LpProblem("Production", pulp.LpMaximize)
x1 = pulp.LpVariable("x1", lowBound=0)
x2 = pulp.LpVariable("x2", lowBound=0)
model += 4*x1 + 5*x2, "profit"
model += x1 + 2*x2 <= 5, "material_A"
model += 2*x1 + 2*x2 <= 8, "material_B"
model += x1 + x2 <= 10, "material_C"
print(model)
model.solve(pulp.HiGHS(msg=False))
status = pulp.LpStatus[model.status]
print("状態:", status)
if status == "Optimal":
    print("製品1:", pulp.value(x1))
    print("製品2:", pulp.value(x2))
    print("利益（万円）:", pulp.value(model.objective))
else:
    print("最適解として読める状態ではありません。条件とログを確認してください。")

## 条件分岐を自分で書く：if・elif・else

07で使った `if status == "Optimal":` を一歩進めます。条件を上から調べ、最初に真になった枝だけを実行します。どれも真でなければ `else` を実行します。`=` は代入、`==` は等しいかの比較です。コロンと字下げに注意します。

In [ ]:
supply_total = 15
demand_total = 13
if supply_total < demand_total:
    print("供給不足")
elif supply_total == demand_total:
    print("合計が一致")
else:
    print("供給に余裕")

### やってみよう：条件分岐
供給の合計は15のまま、上の `demand_total` を13、15、17に変えてください。表示を予想してから、そのセル全体を実行します。

### 解答：条件分岐
13なら `else` の「供給に余裕」、15なら `elif` の「合計が一致」、17なら `if` の「供給不足」です。一つの枝を実行すると残りの枝は実行しません。ここでは合計量だけを比較しています。経路に制限がある一般の場合は、供給が十分でも実行不能になり得ます。

## 3. 輸送問題：誰から誰へ、いくつ送るか

2倉庫から3消費地へ送ります。輸送量は連続量、数量の単位はt、費用係数は千円/tです。すべての経路を使え、輸送量に比例して費用が増えると仮定します。

| 倉庫 | 消費地0 | 消費地1 | 消費地2 | 供給上限 |
|---|---:|---:|---:|---:|
| 0 | 2 | 5 | 6 | 7 |
| 1 | 4 | 1 | 3 | 8 |
| 需要 | 4 | 3 | 6 | — |

ここでは数学もPythonも倉庫$i=0,1$、消費地$j=0,1,2$で統一します。

In [ ]:
supply = [7, 8]
demand = [4, 3, 6]
cost = [[2, 5, 6], [4, 1, 3]]
print(cost[0][2])

`cost[0][2]` は倉庫0から消費地2への単位費用6です。二重リストの外側が倉庫、内側が消費地です。

In [ ]:
for i in range(2):
    for j in range(3):
        print("倉庫", i, "消費地", j, "単位費用", cost[i][j])

### forの中でifを使う
二重ループで各経路を調べ、単位費用が3以下の経路だけ表示します。`print` をさらに字下げし、条件が真のときだけ実行します。

In [ ]:
for i in range(2):
    for j in range(3):
        if cost[i][j] <= 3:
            print(i, j, cost[i][j])

出力は `(0, 0, 2)`、`(1, 1, 1)`、`(1, 2, 3)` です。表示する経路を選んだだけで、他の経路を輸送モデルから除いたわけではありません。後の描画コードも、`if v > 1e-7:` で正の輸送量がある経路だけ描きます。

`range(2)` は0・1、`range(3)` は0・1・2です。内側のループが3回動いてから、外側の添字が次へ進みます。

### やってみよう1
倉庫1から消費地0への単位費用を取り出してください。

In [ ]:
# costの添字を書き、printしてください。

### 解答1
`print(cost[1][0])` で4が表示されます。

## 4. 決定変数と制約

$x_{ij}$ を倉庫$i$から消費地$j$への輸送量とします。

$$\min\ \sum_{i=0}^{1}\sum_{j=0}^{2}c_{ij}x_{ij}$$
$$\sum_{j=0}^{2}x_{ij}\le a_i\quad(i=0,1),\qquad\sum_{i=0}^{1}x_{ij}\ge b_j\quad(j=0,1,2),\qquad x_{ij}\ge0$$

各倉庫の行の和が供給、各消費地の列の和が到着量です。需要は「少なくとも必要な量」として不等式にしています。費用がすべて正の今回のモデルでは、余分に送ると高くなるので、最適解で到着量は需要に一致します。

## 5. 表を使ってPuLPに記述する

`[式 for j in range(3)]` は3個の要素を持つリストを作る書き方です。変数名の `f"x_{i}_{j}"` は添字を文字列へ入れます。`lpSum` は式を足し合わせます。

### 1行から、行と列の表へ

```python
[j for j in range(3)]
# [0, 1, 2]
[[10*i+j for j in range(3)] for i in range(2)]
# [[0, 1, 2], [10, 11, 12]]
```

内側の `j` が列を作り、外側の `i` が行を並べます。次のセルでは数値の代わりに非負のPuLP変数を入れ、2行3列の表を作ります。例えば `i=1, j=2` の変数名は `x_1_2` です。

目的関数の `lpSum(... for i ... for j ...)` は、全6組の費用を一つに足します。制約では、外側の `for i` で倉庫を固定して行を足し、外側の `for j` で消費地を固定して列を足します。

In [ ]:
transport = pulp.LpProblem("Transport", pulp.LpMinimize)
x = [[pulp.LpVariable(f"x_{i}_{j}", lowBound=0)
      for j in range(3)] for i in range(2)]
transport += pulp.lpSum(cost[i][j] * x[i][j]
                        for i in range(2) for j in range(3))
for i in range(2):
    transport += pulp.lpSum(x[i][j] for j in range(3)) <= supply[i]
for j in range(3):
    transport += pulp.lpSum(x[i][j] for i in range(2)) >= demand[j]
transport.solve(pulp.HiGHS(msg=False))
status = pulp.LpStatus[transport.status]
print("状態:", status)
if status == "Optimal":
    flows = [[pulp.value(x[i][j]) for j in range(3)] for i in range(2)]
    for i in range(2):
        print(f"倉庫{i}から [消費地0, 1, 2]:", flows[i])
    print("総費用:", pulp.value(transport.objective))
    print("倉庫の残量:", [supply[i] - sum(flows[i]) for i in range(2)])
    print("消費地への到着量:", [sum(flows[i][j] for i in range(2)) for j in range(3)])
else:
    print("数値を最適解として使わず、供給と需要の条件を確認してください。")

解答：倉庫0から `[4, 0, 1]`、倉庫1から `[0, 3, 5]`、総費用32千円。供給の残量は `[2, 0]` です。すべての需要を満たし、供給上限以下であることを出力で確認してください。

## 6. 結果を図で確認する

In [ ]:
import matplotlib.pyplot as plt
if status == "Optimal":
    fig, ax = plt.subplots(figsize=(8,4.5))
    src = [(0,2),(0,0)]; dst = [(4,2.5),(4,1),(4,-.5)]
    for i,(a,b) in enumerate(src):
        ax.scatter(a,b,s=300,c="#0017c1");ax.text(a-.65,b+.35,f"Warehouse {i}")
    for j,(a,b) in enumerate(dst):
        ax.scatter(a,b,s=250,c="#c06900",marker="s");ax.text(a+.15,b,f"City {j}")
    for i in range(2):
        for j in range(3):
            v=flows[i][j]
            if v>1e-7:
                a,b=src[i];c,d=dst[j]
                ax.annotate("",xy=(c,d),xytext=(a,b),arrowprops=dict(arrowstyle="->",lw=1+v*.6,color="#16764b",alpha=.8))
                f=.32 if i==0 else .68
                ax.text(a+(c-a)*f,b+(d-b)*f,f"{v:g} t",bbox=dict(fc="white",ec="none",alpha=.9))
    ax.set(xlim=(-1,5.2),ylim=(-1,3.2),title=f"Optimal transport / cost = {pulp.value(transport.objective):g}")
    ax.axis("off");plt.show()

### やってみよう2
消費地2の需要を6から10に変更すると、全需要は17tです。供給上限の合計15tで満たせますか。下を実行する前に考えてください。

In [ ]:
print("供給の合計:", sum(supply))
changed_demand = [4, 3, 10]
print("変更後の需要の合計:", sum(changed_demand))

# 元のモデルには追加せず、変更後の需要で新しいモデルを作る。
changed = pulp.LpProblem("Transport_changed_demand", pulp.LpMinimize)
changed_x = [[pulp.LpVariable(f"changed_x_{i}_{j}", lowBound=0)
              for j in range(3)] for i in range(2)]
changed += pulp.lpSum(cost[i][j] * changed_x[i][j]
                      for i in range(2) for j in range(3))
for i in range(2):
    changed += pulp.lpSum(changed_x[i][j] for j in range(3)) <= supply[i]
for j in range(3):
    changed += pulp.lpSum(changed_x[i][j] for i in range(2)) >= changed_demand[j]
changed.solve(pulp.HiGHS(msg=False))
changed_status = pulp.LpStatus[changed.status]
print("変更後の状態:", changed_status)
if changed_status == "Optimal":
    print("変更後の最小費用:", pulp.value(changed.objective))
else:
    print("最適解として使える輸送量は表示しません。供給と需要を確認してください。")

### 解答2
全供給15t < 全需要17tなので実行不能です。この例では経路の制限はありません。一般に経路ごとの上限や通行禁止がある場合は、全供給が足りても実行不能になることがあります。

## 7. 簡易ドッキング：形を変えずに目標点へ合わせる

ここからは採点しない体験です。2次元の3点を剛体分子、×印を結合部位の目標点に見立てます。点0・1・2の対応をあらかじめ固定します。

全点を同じ量だけ平行移動し、形を変えずに目標へ近づけます。回転は0・90・180・270度の4候補に固定し、各候補で最良の平行移動を求めて比較します。結合エネルギーや立体衝突は扱いません。通常の分子ドッキングそのものではありません。

## 8. 絶対値を不等式で表す

ある回転候補の点を $(a_{ix},a_{iy})$、目標点を $(b_{ix},b_{iy})$、全点共通の平行移動を $(t_x,t_y)$ とします。

横のずれ $d=a_{ix}+t_x-b_{ix}$ に対し、$u_{ix}\ge d$、$u_{ix}\ge-d$ を置けば $u_{ix}\ge|d|$ です。縦も同様にします。

$$\begin{aligned}\min\quad&\sum_{i=0}^2(u_{ix}+u_{iy})\\\text{制約}\quad&u_{ix}\ge a_{ix}+t_x-b_{ix},\quad u_{ix}\ge-(a_{ix}+t_x-b_{ix})\\&u_{iy}\ge a_{iy}+t_y-b_{iy},\quad u_{iy}\ge-(a_{iy}+t_y-b_{iy})\\&u_{ix},u_{iy}\ge0,\quad t_x,t_y\text{ は自由変数}\end{aligned}$$

最小化により、補助変数は各絶対値まで下がります。各向きでの問題はLPです。向きを比べる部分は4候補の列挙です。回転角そのものを連続変数にすると、この定式化はLPではなくなります。

In [ ]:
# 描画と候補の比較のためのコード。ここはそのまま実行します。
import numpy as np
import matplotlib.pyplot as plt
ligand = np.array([[0, 0], [2, 0], [2, 1]], dtype=float)
target = np.array([[3, 2], [3, 4], [2, 4.5]], dtype=float)
rotations = [np.array([[1,0],[0,1]]), np.array([[0,-1],[1,0]]),
             np.array([[-1,0],[0,-1]]), np.array([[0,1],[-1,0]])]
results = []
for angle, rotation in zip([0, 90, 180, 270], rotations):
    a = ligand @ rotation.T
    fit = pulp.LpProblem(f"Pose_{angle}", pulp.LpMinimize)
    tx = pulp.LpVariable("tx", lowBound=None)
    ty = pulp.LpVariable("ty", lowBound=None)
    ux = [pulp.LpVariable(f"ux_{i}", lowBound=0) for i in range(3)]
    uy = [pulp.LpVariable(f"uy_{i}", lowBound=0) for i in range(3)]
    fit += pulp.lpSum(ux) + pulp.lpSum(uy)
    for i in range(3):
        fit += ux[i] >= a[i,0] + tx - target[i,0]
        fit += ux[i] >= -(a[i,0] + tx - target[i,0])
        fit += uy[i] >= a[i,1] + ty - target[i,1]
        fit += uy[i] >= -(a[i,1] + ty - target[i,1])
    fit.solve(pulp.HiGHS(msg=False))
    assert pulp.LpStatus[fit.status] == "Optimal"
    shift = np.array([pulp.value(tx), pulp.value(ty)])
    pose = a + shift
    score = pulp.value(fit.objective)
    # 全原子の平行移動が同じなので、分子内部の距離は変わりません。
    assert np.allclose(np.linalg.norm(pose[:,None]-pose[None,:],axis=-1),
                       np.linalg.norm(ligand[:,None]-ligand[None,:],axis=-1))
    assert np.isclose(score, np.abs(pose-target).sum())
    results.append(dict(angle=angle, shift=shift, pose=pose, score=score))
    print(f"回転 {angle:3}度: 移動 {shift}, ずれ合計 {score:.2f}")
best = min(results, key=lambda r: r["score"])
print("候補内で最良の回転:", best["angle"], "度")
fig, axes = plt.subplots(1,4,figsize=(13,3.6))
for ax,r in zip(axes,results):
    ax.scatter(*target.T, marker="x", s=90, c="#c06900", label="Targets")
    ax.plot(*r["pose"].T,"o-",c="#16764b",lw=3,label="Rigid ligand")
    for i in range(3):
        ax.plot([r["pose"][i,0],target[i,0]], [r["pose"][i,1],target[i,1]], ":",c="#888")
        ax.annotate(str(i),target[i],xytext=(5,5),textcoords="offset points")
    ax.set(xlim=(-1,6),ylim=(0,7),title=f'{r["angle"]} deg / L1 = {r["score"]:.1f}',aspect="equal")
    ax.grid(alpha=.2)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### やってみよう3
4候補の図と出力を見て、ずれが最小の向きと移動量を確認してください。次に `target` のすべての点の横座標へ1を加え、同じセルを再実行するとどうなるか予想してください。

### 解答3
最良は90度、移動量は $(3,2)$、ずれの合計は0.5です。目標点全体を横に1だけ動かすと、最良の移動量も横に1増え、ずれの合計は変わりません。元の座標へ戻してから次へ進んでください。

## 9. 自分で条件を変えて確かめる（任意）

下のデータ用セルに、供給 `[8, 9]`、需要 `[4, 5, 6]`、費用 `[[2, 5, 6], [4, 1, 3]]` を入力して実行してください。次のセルで解き、状態・輸送量・総費用・残量の順に確かめます。採点や提出はありません。


In [ ]:
supply = [8, 9]  # 2倉庫の供給上限
demand = [4, 5, 6]  # 3消費地の需要
cost = [[2, 5, 6], [4, 1, 3]]  # 単位費用


In [ ]:
if supply is None or demand is None or cost is None:
    print("練習の条件を上のデータ用セルへ入力し、先に実行してください。")
else:
    transport = pulp.LpProblem("Transport", pulp.LpMinimize)
    x = [[pulp.LpVariable(f"x_{i}_{j}", lowBound=0)
          for j in range(3)] for i in range(2)]
    transport += pulp.lpSum(cost[i][j] * x[i][j]
                            for i in range(2) for j in range(3))
    for i in range(2):
        transport += pulp.lpSum(x[i][j] for j in range(3)) <= supply[i]
    for j in range(3):
        transport += pulp.lpSum(x[i][j] for i in range(2)) >= demand[j]
    transport.solve(pulp.HiGHS(msg=False))
    status = pulp.LpStatus[transport.status]
    print("状態:", status)
    if status == "Optimal":
        flows = [[pulp.value(x[i][j]) for j in range(3)] for i in range(2)]
        for i in range(2):
            print(f"倉庫{i}から [消費地0, 1, 2]:", flows[i])
        print("総費用:", pulp.value(transport.objective))
        print("倉庫の残量:", [supply[i] - sum(flows[i]) for i in range(2)])
        print("消費地への到着量:", [sum(flows[i][j] for i in range(2)) for j in range(3)])
    else:
        print("数値を最適解として使わず、供給と需要の条件を確認してください。")


### 解答：条件変更
倉庫0から `[4, 0, 2]`、倉庫1から `[0, 5, 4]`。費用は `2×4 + 6×2 + 1×5 + 3×4 = 37` 千円。到着量は `[4, 5, 6]`、供給残量は `[2, 0]` です。需要の最後を10へ増やすと、全需要19tが供給17tを超え、実行不能です。


## まとめ

表の行と列を添字に対応させると、大きい問題も短いコードで書けます。ソルバの出力は、状態・制約・目的値の順に確認します。絶対値のような表現も、補助変数を使うとLPで表せる場合があります。

参考：[PuLP](https://github.com/coin-or/pulp/tree/3.3.0)、[HiGHS](https://highs.dev/)、[通常のドッキングとの違い：AutoDock Vina原著](https://pmc.ncbi.nlm.nih.gov/articles/PMC3041641/)。

## 09・10へのつながり
倉庫と消費地を頂点、輸送経路を有向辺として見るとネットワークになります。09は途中の頂点を通る**最短経路**、10は辺の容量を守る**最大流**です。同じ図でも、辺の数値が費用・長さ・容量のどれかを区別します。Driveへの保存を確認して終了してください。
